In [1]:
import warnings
warnings.filterwarnings("ignore")
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
import os

In [2]:
PANEL_PATH = "../data/processed/student_assessment_panel.csv"
OUT_DIR    = "results/figures"
TABLE_DIR  = "results/tables"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

In [3]:
FEMALE_COLOR = "#E07B54"   # warm orange
MALE_COLOR   = "#4C72B0"   # muted blue
sns.set_theme(style="whitegrid", font_scale=1.15)

In [4]:
print("=" * 65)
print("GENDER x ENGAGEMENT ANALYSIS")
print("=" * 65)
 
df = pd.read_csv(PANEL_PATH)
 
# Ensure required columns exist
required = ["id_student", "student_course_id", "id_assessment",
            "log_clicks_28d", "score", "is_female", "assessment_type"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in panel: {missing}\n"
                     f"Available: {list(df.columns)}")
 
# Drop rows missing key variables
df = df.dropna(subset=required)
 
# Panel entity = student-course
# student_course_id already uniquely identifies the student-course panel
df["entity"] = df["student_course_id"].astype(str)
 
# Engagement squared for nonlinear models
df["log_clicks_28d_sq"] = df["log_clicks_28d"] ** 2
 
# Demeaned engagement within entity (for within-student centering)
df["eng_demean"] = df.groupby("entity")["log_clicks_28d"].transform(
    lambda x: x - x.mean()
)
df["eng_demean_sq"] = df["eng_demean"] ** 2
 
# Engagement deciles (global, so comparable across genders)
df["eng_decile"] = pd.qcut(df["log_clicks_28d"], q=10,
                            labels=False, duplicates="drop")
 
# Moderate engagement flag: deciles 2-7 (0-indexed), i.e. the middle range
df["is_moderate_eng"] = df["eng_decile"].between(2, 7).astype(int)
 
print(f"\nSample: {len(df):,} observations | "
      f"{df['entity'].nunique():,} student-course panels | "
      f"{df['id_assessment'].nunique()} assessments")
print(f"Gender split: "
      f"{df['is_female'].mean()*100:.1f}% female | "
      f"{(1-df['is_female']).mean()*100:.1f}% male\n")

GENDER x ENGAGEMENT ANALYSIS

Sample: 171,241 observations | 23,322 student-course panels | 188 assessments
Gender split: 46.1% female | 53.9% male



In [5]:
### HELPER FUNCTIONS

def run_fe(data, formula_vars, entity_col="entity", time_col="id_assessment",
           cluster_col="entity"):
    """
    Fit a two-way fixed-effects model (entity + time FE) via PanelOLS.
 
    formula_vars : list of column names to use as regressors (right-hand side)
    Returns the fitted result object.
    """
    df_fe = data.copy()
    df_fe = df_fe.set_index([entity_col, time_col])
    df_fe = df_fe.dropna(subset=formula_vars + ["score"])
 
    import linearmodels.panel as lm
    mod = lm.PanelOLS(
        dependent=df_fe["score"],
        exog=df_fe[formula_vars],
        entity_effects=True,
        time_effects=True,
    )
    res = mod.fit(cov_type="clustered", cluster_entity=True, debiased=True)
    return res
 
 
def print_result_table(res, title, save_path=None):
    """Pretty-print a linearmodels result and optionally save to CSV."""
    params = res.params
    se     = res.std_errors
    tstat  = res.tstats
    pval   = res.pvalues
    ci_lo  = res.params - 1.96 * res.std_errors
    ci_hi  = res.params + 1.96 * res.std_errors
 
    tbl = pd.DataFrame({
        "coef":   params.round(4),
        "se":     se.round(4),
        "t":      tstat.round(3),
        "p":      pval.round(4),
        "ci_lo":  ci_lo.round(4),
        "ci_hi":  ci_hi.round(4),
    })
 
    print(f"\n{'─'*65}")
    print(f"  {title}")
    print(f"  N = {int(res.nobs):,}  |  R² (within) = {res.rsquared:.4f}")
    print(f"{'─'*65}")
    print(tbl.to_string())
    print(f"{'─'*65}\n")
 
    if save_path:
        tbl.to_csv(save_path)
        print(f"  Saved to {save_path}")
 
    return tbl

In [6]:
# INVESTIGATION 1: NONLINEAR GENDER INTERACTION
print("\n" + "═"*65)
print("INVESTIGATION 1: NONLINEAR GENDER INTERACTION")
print("═"*65)
 
print("""
Motivation
----------
A standard linear interaction (engagement x is_female) tests only whether
the slope differs between genders. The observed pattern -- convergence at
both tails, divergence in the middle -- suggests the CURVATURE of the
engagement-score relationship differs by gender. We test this by adding
a quadratic interaction term.
 
Model A (baseline linear interaction):
  score = b1*eng + b2*female + b3*(eng x female) + entity_FE + assessment_FE
 
Model B (quadratic interaction):
  score = b1*eng + b2*eng^2 + b3*female
        + b4*(eng x female) + b5*(eng^2 x female)
        + entity_FE + assessment_FE
 
Key tests:
  - b3 in Model A: does slope differ?
  - b5 in Model B: does curvature differ?
  - Joint F-test on (b4, b5) in Model B: any gender moderation?
""")
 
# -- Model A: linear interaction
df["eng_x_female"] = df["log_clicks_28d"] * df["is_female"]
 
# is_female is time-invariant -> perfectly absorbed by entity FE
# We only include the interaction term; the main female effect is not identified
resA = run_fe(
    df,
    ["log_clicks_28d", "eng_x_female"]
)
tblA = print_result_table(
    resA,
    "Model A: Linear gender interaction",
    save_path=f"{TABLE_DIR}/gender_model_A_linear.csv"
)
 
# -- Model B: quadratic interaction
df["eng_sq_x_female"] = df["log_clicks_28d_sq"] * df["is_female"]
 
resB = run_fe(
    df,
    ["log_clicks_28d", "log_clicks_28d_sq",
     "eng_x_female", "eng_sq_x_female"],
)
tblB = print_result_table(
    resB,
    "Model B: Quadratic gender interaction",
    save_path=f"{TABLE_DIR}/gender_model_B_quadratic.csv"
)
 
# Interpretation
b_eng       = resB.params["log_clicks_28d"]
b_eng_sq    = resB.params["log_clicks_28d_sq"]
b_eng_f     = resB.params["eng_x_female"]
b_eng_sq_f  = resB.params["eng_sq_x_female"]
p_eng_f     = resB.pvalues["eng_x_female"]
p_eng_sq_f  = resB.pvalues["eng_sq_x_female"]
 
print("Interpretation of Model B:")
print("  Note: main female effect absorbed by entity FE (time-invariant).")
print(f"  Male baseline slope:    {b_eng:.3f}*eng + {b_eng_sq:.4f}*eng^2")
print(f"  Female differential:  + {b_eng_f:.3f}*eng + {b_eng_sq_f:.4f}*eng^2")
print(f"  => Female slope total:  {b_eng+b_eng_f:.3f}*eng + {b_eng_sq+b_eng_sq_f:.4f}*eng^2")
print(f"  Linear interaction    p = {p_eng_f:.4f}  {'*' if p_eng_f<0.05 else '(ns)'}")
print(f"  Quadratic interaction p = {p_eng_sq_f:.4f}  {'*' if p_eng_sq_f<0.05 else '(ns)'}")
 
# Plot predicted curves from Model B over observed engagement range
eng_range = np.linspace(df["log_clicks_28d"].quantile(0.05),
                        df["log_clicks_28d"].quantile(0.95), 200)
 
pred_male   = b_eng * eng_range + b_eng_sq * eng_range**2
pred_female = (b_eng + b_eng_f) * eng_range + (b_eng_sq + b_eng_sq_f) * eng_range**2
 
# Centre both at zero so we're comparing shapes, not levels
pred_male   -= pred_male.mean()
pred_female -= pred_female.mean()
 
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(eng_range, pred_male,   color=MALE_COLOR,   lw=2.5, label="Male (predicted)")
ax.plot(eng_range, pred_female, color=FEMALE_COLOR, lw=2.5, label="Female (predicted)")
ax.axhline(0, color="grey", lw=0.8, linestyle="--")
ax.set_xlabel("Pre-assessment engagement (log(1 + clicks 28d))")
ax.set_ylabel("Predicted score deviation (demeaned)")
ax.set_title("Investigation 1: Predicted engagement-score curves by gender\n"
             "(within-student fixed effects, quadratic specification)")
ax.legend()
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/gender_inv1_predicted_curves.png", dpi=150)
plt.close()
print(f"\n  Figure saved: {OUT_DIR}/gender_inv1_predicted_curves.png")


═════════════════════════════════════════════════════════════════
INVESTIGATION 1: NONLINEAR GENDER INTERACTION
═════════════════════════════════════════════════════════════════

Motivation
----------
A standard linear interaction (engagement x is_female) tests only whether
the slope differs between genders. The observed pattern -- convergence at
both tails, divergence in the middle -- suggests the CURVATURE of the
engagement-score relationship differs by gender. We test this by adding
a quadratic interaction term.

Model A (baseline linear interaction):
  score = b1*eng + b2*female + b3*(eng x female) + entity_FE + assessment_FE

Model B (quadratic interaction):
  score = b1*eng + b2*eng^2 + b3*female
        + b4*(eng x female) + b5*(eng^2 x female)
        + entity_FE + assessment_FE

Key tests:
  - b3 in Model A: does slope differ?
  - b5 in Model B: does curvature differ?
  - Joint F-test on (b4, b5) in Model B: any gender moderation?


───────────────────────────────────────────

In [7]:
# INVESTIGATION 2: DECILE-LEVEL MEANS WITH CONFIDENCE INTERVALS
print("\n" + "═"*65)
print("INVESTIGATION 2: DECILE MEANS WITH CONFIDENCE INTERVALS")
print("═"*65)
 
print("""
Motivation
----------
A smoothed trend line can exaggerate or suppress divergence depending on
bandwidth. Computing mean scores per engagement decile separately for each
gender, with 95% CIs, lets us see exactly WHICH deciles drive the divergence
and whether it is statistically reliable.
""")
 
decile_stats = (
    df.groupby(["eng_decile", "is_female"])["score"]
    .agg(mean="mean", se=lambda x: x.sem(), n="count")
    .reset_index()
)
decile_stats["ci_lo"] = decile_stats["mean"] - 1.96 * decile_stats["se"]
decile_stats["ci_hi"] = decile_stats["mean"] + 1.96 * decile_stats["se"]
decile_stats["gender"] = decile_stats["is_female"].map({1: "Female", 0: "Male"})
 
# Compute gap per decile and its significance
gap_rows = []
for dec in sorted(df["eng_decile"].dropna().unique()):
    sub = df[df["eng_decile"] == dec]
    f_scores = sub.loc[sub["is_female"] == 1, "score"].dropna()
    m_scores = sub.loc[sub["is_female"] == 0, "score"].dropna()
    if len(f_scores) < 5 or len(m_scores) < 5:
        continue
    gap  = f_scores.mean() - m_scores.mean()
    tval, pval = stats.ttest_ind(f_scores, m_scores, equal_var=False)
    gap_rows.append({
        "decile": int(dec),
        "female_mean": round(f_scores.mean(), 3),
        "male_mean":   round(m_scores.mean(), 3),
        "gap (F-M)":   round(gap, 3),
        "t":           round(tval, 3),
        "p":           round(pval, 4),
        "sig":         "*" if pval < 0.05 else ""
    })
 
gap_df = pd.DataFrame(gap_rows)
print("  Gender gap by engagement decile (naive t-test, not FE-adjusted):")
print(gap_df.to_string(index=False))
gap_df.to_csv(f"{TABLE_DIR}/gender_gap_by_decile.csv", index=False)
print(f"\n  Saved: {TABLE_DIR}/gender_gap_by_decile.csv")
 
# Plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
 
# Left: mean + CI
ax = axes[0]
for gender, color in [("Female", FEMALE_COLOR), ("Male", MALE_COLOR)]:
    sub = decile_stats[decile_stats["gender"] == gender]
    ax.plot(sub["eng_decile"], sub["mean"], marker="o", color=color,
            lw=2, label=gender)
    ax.fill_between(sub["eng_decile"], sub["ci_lo"], sub["ci_hi"],
                    alpha=0.18, color=color)
ax.set_xlabel("Engagement decile (0 = lowest)")
ax.set_ylabel("Average assessment score")
ax.set_title("Mean score by engagement decile\nwith 95% CI (descriptive)")
ax.legend()
 
# Right: gap per decile with significance markers
ax2 = axes[1]
colors = [FEMALE_COLOR if g > 0 else MALE_COLOR for g in gap_df["gap (F-M)"]]
bars = ax2.bar(gap_df["decile"], gap_df["gap (F-M)"], color=colors, alpha=0.75)
ax2.axhline(0, color="black", lw=0.8)
for _, row in gap_df[gap_df["sig"] == "*"].iterrows():
    ax2.text(row["decile"], row["gap (F-M)"] + 0.1 * np.sign(row["gap (F-M)"]),
             "*", ha="center", fontsize=14)
ax2.set_xlabel("Engagement decile")
ax2.set_ylabel("Female minus Male mean score")
ax2.set_title("Gender gap by decile\n(* = p < 0.05, naive t-test)")
 
fig.tight_layout()
fig.savefig(f"{OUT_DIR}/gender_inv2_decile_CIs.png", dpi=150)
plt.close()
print(f"  Figure saved: {OUT_DIR}/gender_inv2_decile_CIs.png")
 


═════════════════════════════════════════════════════════════════
INVESTIGATION 2: DECILE MEANS WITH CONFIDENCE INTERVALS
═════════════════════════════════════════════════════════════════

Motivation
----------
A smoothed trend line can exaggerate or suppress divergence depending on
bandwidth. Computing mean scores per engagement decile separately for each
gender, with 95% CIs, lets us see exactly WHICH deciles drive the divergence
and whether it is statistically reliable.

  Gender gap by engagement decile (naive t-test, not FE-adjusted):
 decile  female_mean  male_mean  gap (F-M)      t      p sig
      0       69.315     67.229      2.086  6.362 0.0000   *
      1       74.084     67.170      6.914 20.673 0.0000   *
      2       75.418     70.334      5.083 16.010 0.0000   *
      3       76.074     72.943      3.131 10.370 0.0000   *
      4       77.234     74.111      3.123 10.566 0.0000   *
      5       77.722     76.662      1.060  3.810 0.0001   *
      6       78.347     7

In [8]:
# INVESTIGATION 3: IS THE GAP CONCENTRATED IN MODERATE ENGAGEMENT?
print("\n" + "═"*65)
print("INVESTIGATION 3: MODERATE ENGAGEMENT CONCENTRATION TEST")
print("═"*65)
 
print("""
Motivation
----------
If the curves converge at both tails and diverge in the middle, the gender
difference in engagement returns is specifically a moderate-engagement
phenomenon. We test this by running the fixed-effects model separately on
three engagement bands: low (deciles 0-2), moderate (deciles 3-6), high (7-9).
 
If the pattern is real:
  - Low and high bands: gender interaction near zero
  - Moderate band: gender interaction significantly different from zero
""")
 
bands = {
    "Low (deciles 0-2)":      df["eng_decile"].between(0, 2),
    "Moderate (deciles 3-6)": df["eng_decile"].between(3, 6),
    "High (deciles 7-9)":     df["eng_decile"].between(7, 9),
}
 
band_results = []
for band_name, mask in bands.items():
    sub = df[mask].copy()
    sub["eng_x_female"] = sub["log_clicks_28d"] * sub["is_female"]
    n = len(sub)
    if n < 200:
        print(f"  Skipping {band_name}: too few observations ({n})")
        continue
    try:
        res = run_fe(sub, ["log_clicks_28d", "eng_x_female"])
        coef = res.params.get("eng_x_female", np.nan)
        se   = res.std_errors.get("eng_x_female", np.nan)
        pval = res.pvalues.get("eng_x_female", np.nan)
        band_results.append({
            "Band":          band_name,
            "N":             n,
            "eng_x_female":  round(coef, 4),
            "SE":            round(se,   4),
            "p":             round(pval, 4),
            "sig":           "*" if pval < 0.05 else ""
        })
        print(f"  {band_name}: coef={coef:.4f}, SE={se:.4f}, p={pval:.4f} "
              f"{'*' if pval<0.05 else '(ns)'}")
    except Exception as e:
        print(f"  {band_name}: model failed ({e})")
 
band_df = pd.DataFrame(band_results)
band_df.to_csv(f"{TABLE_DIR}/gender_band_interactions.csv", index=False)
print(f"\n  Saved: {TABLE_DIR}/gender_band_interactions.csv")
 
# Also run a three-way interaction: engagement x female x moderate_band
print("\n  Three-way interaction model:")
print("  score = eng + eng*female + eng*moderate + eng*female*moderate + FE")
 
df["eng_x_mod"]        = df["log_clicks_28d"] * df["is_moderate_eng"]
df["eng_x_fem_x_mod"]  = df["log_clicks_28d"] * df["is_female"] * df["is_moderate_eng"]
df["eng_x_female"]     = df["log_clicks_28d"] * df["is_female"]   # ensure present
 
# is_female and is_moderate_eng are absorbed; keep only interactions
res3way = run_fe(
    df,
    ["log_clicks_28d", "eng_x_female", "eng_x_mod", "eng_x_fem_x_mod"]
)
tbl3 = print_result_table(
    res3way,
    "3-way interaction: engagement x female x moderate-band",
    save_path=f"{TABLE_DIR}/gender_3way_interaction.csv"
)
 
# Visualise band-level interaction coefficients
if band_results:
    fig, ax = plt.subplots(figsize=(8, 4))
    x = np.arange(len(band_df))
    colors_band = [MALE_COLOR if p >= 0.05 else FEMALE_COLOR
                   for p in band_df["p"]]
    ax.bar(x, band_df["eng_x_female"], color=colors_band, alpha=0.8,
           yerr=1.96*band_df["SE"], capsize=5)
    ax.axhline(0, color="black", lw=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(band_df["Band"], fontsize=10)
    ax.set_ylabel("Interaction coefficient\n(engagement x female)")
    ax.set_title("Investigation 3: Gender interaction coefficient by engagement band\n"
                 "(orange = p < 0.05)")
    for i, row in band_df.iterrows():
        if row["sig"] == "*":
            ax.text(i, row["eng_x_female"] + 0.05, "*", ha="center", fontsize=14)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/gender_inv3_band_coefficients.png", dpi=150)
    plt.close()
    print(f"  Figure saved: {OUT_DIR}/gender_inv3_band_coefficients.png")


═════════════════════════════════════════════════════════════════
INVESTIGATION 3: MODERATE ENGAGEMENT CONCENTRATION TEST
═════════════════════════════════════════════════════════════════

Motivation
----------
If the curves converge at both tails and diverge in the middle, the gender
difference in engagement returns is specifically a moderate-engagement
phenomenon. We test this by running the fixed-effects model separately on
three engagement bands: low (deciles 0-2), moderate (deciles 3-6), high (7-9).

If the pattern is real:
  - Low and high bands: gender interaction near zero
  - Moderate band: gender interaction significantly different from zero

  Low (deciles 0-2): coef=-0.2802, SE=0.2270, p=0.2170 (ns)
  Moderate (deciles 3-6): coef=-0.0461, SE=0.4617, p=0.9205 (ns)
  High (deciles 7-9): coef=0.0982, SE=0.5260, p=0.8519 (ns)

  Saved: results/tables/gender_band_interactions.csv

  Three-way interaction model:
  score = eng + eng*female + eng*moderate + eng*female*moderate + F

In [9]:
# INVESTIGATION 4: ASSESSMENT TYPE COMPOSITION CHECK
print("\n" + "═"*65)
print("INVESTIGATION 4: ASSESSMENT TYPE COMPOSITION CHECK")
print("═"*65)
 
print("""
Motivation
----------
If male and female students in this sample take different mixes of TMAs,
CMAs, and Exams, and those types have different engagement-score curves,
then the aggregate gender x engagement pattern may be purely compositional.
 
Check 1: Does the gender mix differ across assessment types?
Check 2: Run the gender x engagement interaction WITHIN each assessment type.
         If the divergence disappears, it is compositional.
         If it persists in at least one type, it is behavioral.
""")
 
# Check 1: gender composition by assessment type
comp = (df.groupby("assessment_type")["is_female"]
        .agg(female_share="mean", n="count")
        .reset_index())
comp["female_pct"] = (comp["female_share"] * 100).round(1)
print("  Gender composition by assessment type:")
print(comp[["assessment_type", "female_pct", "n"]].to_string(index=False))
 
# Chi-square test: is gender distribution equal across types?
ct = pd.crosstab(df["assessment_type"], df["is_female"])
chi2, p_chi, dof, _ = stats.chi2_contingency(ct)
print(f"\n  Chi-square test (gender vs. type): chi2={chi2:.2f}, p={p_chi:.4f}, dof={dof}")
print(f"  {'Gender composition differs significantly across types.' if p_chi<0.05 else 'No significant composition difference.'}\n")
 
# Check 2: interaction within each assessment type
type_results = []
for atype in df["assessment_type"].dropna().unique():
    sub = df[df["assessment_type"] == atype].copy()
    sub["eng_x_female"] = sub["log_clicks_28d"] * sub["is_female"]
    n = len(sub)
    if n < 100:
        continue
    try:
        res = run_fe(sub, ["log_clicks_28d", "eng_x_female"])
        coef = res.params.get("eng_x_female", np.nan)
        se   = res.std_errors.get("eng_x_female", np.nan)
        pval = res.pvalues.get("eng_x_female", np.nan)
        eng_coef = res.params.get("log_clicks_28d", np.nan)
        type_results.append({
            "assessment_type":  atype,
            "N":                n,
            "main_eng_coef":    round(eng_coef, 4),
            "eng_x_female":     round(coef, 4),
            "SE":               round(se,   4),
            "p":                round(pval, 4),
            "sig":              "*" if pval < 0.05 else ""
        })
    except Exception as e:
        print(f"  {atype}: model failed ({e})")
 
type_df = pd.DataFrame(type_results)
print("  Gender x engagement interaction by assessment type:")
print(type_df.to_string(index=False))
type_df.to_csv(f"{TABLE_DIR}/gender_interaction_by_type.csv", index=False)
print(f"\n  Saved: {TABLE_DIR}/gender_interaction_by_type.csv")
 
# Plot: side-by-side for each assessment type
if len(type_df) > 0:
    fig, axes = plt.subplots(1, len(type_df), figsize=(5*len(type_df), 5),
                             sharey=False)
    if len(type_df) == 1:
        axes = [axes]
 
    for ax, (_, row) in zip(axes, type_df.iterrows()):
        atype = row["assessment_type"]
        sub = df[df["assessment_type"] == atype]
        ds = (sub.groupby(["eng_decile", "is_female"])["score"]
              .mean().reset_index())
        ds["gender"] = ds["is_female"].map({1: "Female", 0: "Male"})
        for g, c in [("Female", FEMALE_COLOR), ("Male", MALE_COLOR)]:
            gd = ds[ds["gender"] == g]
            ax.plot(gd["eng_decile"], gd["score"], marker="o",
                    color=c, lw=2, label=g)
        ax.set_title(f"{atype}\ninteraction coef = {row['eng_x_female']:.3f} "
                     f"({'p=' + str(row['p']) if row['sig']=='*' else 'ns'})")
        ax.set_xlabel("Engagement decile")
        ax.set_ylabel("Mean score")
        ax.legend(fontsize=9)
 
    fig.suptitle("Investigation 4: Gender x engagement by assessment type",
                 fontsize=13, y=1.02)
    fig.tight_layout()
    fig.savefig(f"{OUT_DIR}/gender_inv4_by_assessment_type.png",
                dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Figure saved: {OUT_DIR}/gender_inv4_by_assessment_type.png")


═════════════════════════════════════════════════════════════════
INVESTIGATION 4: ASSESSMENT TYPE COMPOSITION CHECK
═════════════════════════════════════════════════════════════════

Motivation
----------
If male and female students in this sample take different mixes of TMAs,
CMAs, and Exams, and those types have different engagement-score curves,
then the aggregate gender x engagement pattern may be purely compositional.

Check 1: Does the gender mix differ across assessment types?
Check 2: Run the gender x engagement interaction WITHIN each assessment type.
         If the divergence disappears, it is compositional.
         If it persists in at least one type, it is behavioral.

  Gender composition by assessment type:
assessment_type  female_pct     n
            CMA        45.8 69975
           Exam        34.7  4955
            TMA        46.8 96311

  Chi-square test (gender vs. type): chi2=283.66, p=0.0000, dof=2
  Gender composition differs significantly across types.

  Ex

In [10]:
# SUMMARY DASHBOARD FIGURE
print("\n" + "═"*65)
print("GENERATING SUMMARY DASHBOARD")
print("═"*65)
 
fig = plt.figure(figsize=(15, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.35)
 
# ─ Panel A: raw decile trends with CI ─────────────────────────────────────────
ax_a = fig.add_subplot(gs[0, 0])
for gender, color in [("Female", FEMALE_COLOR), ("Male", MALE_COLOR)]:
    sub = decile_stats[decile_stats["gender"] == gender]
    ax_a.plot(sub["eng_decile"], sub["mean"], marker="o", color=color,
              lw=2, label=gender)
    ax_a.fill_between(sub["eng_decile"], sub["ci_lo"], sub["ci_hi"],
                      alpha=0.15, color=color)
ax_a.set_title("A. Mean score by decile\n(descriptive, 95% CI)")
ax_a.set_xlabel("Engagement decile")
ax_a.set_ylabel("Mean score")
ax_a.legend(fontsize=9)
 
# ─ Panel B: gender gap per decile ─────────────────────────────────────────────
ax_b = fig.add_subplot(gs[0, 1])
clrs = [FEMALE_COLOR if g > 0 else MALE_COLOR for g in gap_df["gap (F-M)"]]
ax_b.bar(gap_df["decile"], gap_df["gap (F-M)"], color=clrs, alpha=0.75)
ax_b.axhline(0, color="black", lw=0.8)
for _, row in gap_df[gap_df["sig"] == "*"].iterrows():
    ax_b.text(row["decile"], row["gap (F-M)"] + 0.1*np.sign(row["gap (F-M)"]),
              "*", ha="center", fontsize=13)
ax_b.set_title("B. Female - Male gap\nper decile (* p<0.05)")
ax_b.set_xlabel("Engagement decile")
ax_b.set_ylabel("Score gap (F - M)")
 
# ─ Panel C: predicted curves from quadratic FE model ─────────────────────────
ax_c = fig.add_subplot(gs[0, 2])
ax_c.plot(eng_range, pred_male,   color=MALE_COLOR,   lw=2.5, label="Male")
ax_c.plot(eng_range, pred_female, color=FEMALE_COLOR, lw=2.5, label="Female")
ax_c.axhline(0, color="grey", lw=0.7, linestyle="--")
ax_c.set_title("C. Predicted curves (FE)\nQuadratic specification")
ax_c.set_xlabel("log(1 + clicks 28d)")
ax_c.set_ylabel("Score deviation (demeaned)")
ax_c.legend(fontsize=9)
 
# ─ Panel D: band-level interaction coefficients ───────────────────────────────
ax_d = fig.add_subplot(gs[1, 0])
if len(band_df) > 0:
    x = np.arange(len(band_df))
    clrs_b = [FEMALE_COLOR if p < 0.05 else "#AAAAAA" for p in band_df["p"]]
    ax_d.bar(x, band_df["eng_x_female"], color=clrs_b, alpha=0.8,
             yerr=1.96*band_df["SE"], capsize=5)
    ax_d.axhline(0, color="black", lw=0.8)
    ax_d.set_xticks(x)
    short_labels = [b.split("(")[0].strip() for b in band_df["Band"]]
    ax_d.set_xticklabels(short_labels, fontsize=9)
    ax_d.set_title("D. Interaction coef\nby engagement band")
    ax_d.set_ylabel("Coef (eng x female)")
else:
    ax_d.text(0.5, 0.5, "No band results", ha="center", va="center")
    ax_d.set_title("D. Band-level interactions")
 
# ─ Panel E: by assessment type ────────────────────────────────────────────────
ax_e = fig.add_subplot(gs[1, 1:])
if len(type_df) > 0:
    for atype, color in zip(type_df["assessment_type"],
                             [MALE_COLOR, FEMALE_COLOR, "#2ca02c"]):
        sub_ds = (df[df["assessment_type"] == atype]
                  .groupby(["eng_decile", "is_female"])["score"]
                  .mean().reset_index())
        for gval, lstyle in [(1, "-"), (0, "--")]:
            gd = sub_ds[sub_ds["is_female"] == gval]
            lbl = f"{atype} ({'F' if gval==1 else 'M'})"
            ax_e.plot(gd["eng_decile"], gd["score"], linestyle=lstyle,
                      color=color, lw=1.8, label=lbl, marker="o", markersize=4)
    ax_e.set_title("E. Gender patterns within each assessment type\n"
                   "(solid=Female, dashed=Male)")
    ax_e.set_xlabel("Engagement decile")
    ax_e.set_ylabel("Mean score")
    ax_e.legend(fontsize=8, ncol=2)
else:
    ax_e.text(0.5, 0.5, "No type results", ha="center", va="center")
 
fig.suptitle("Gender x Engagement Deep-Dive: All Investigations",
             fontsize=14, fontweight="bold", y=1.01)
 
fig.savefig(f"{OUT_DIR}/gender_interaction_dashboard.png",
            dpi=150, bbox_inches="tight")
plt.close()
print(f"  Dashboard saved: {OUT_DIR}/gender_interaction_dashboard.png")
 


═════════════════════════════════════════════════════════════════
GENERATING SUMMARY DASHBOARD
═════════════════════════════════════════════════════════════════
  Dashboard saved: results/figures/gender_interaction_dashboard.png
